# NFL ESPN Fantasy scrapping

## Setup

In [1]:
import requests
import json

from time import sleep
from random import random

from math import nan

import os
import polars as pl

from typing import TypedDict

In [ ]:
MAP_POSITION_SLOT = {
    "ALL": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 23, 24],
    "QB": [0],
    "RB": [2],
    "WR": [4],
    "TE": [6],
    "FLEX": [23], # RB/WR/TE
    "D/ST": [16],
    "K": [17],
    "HC": [19],
}

MAP_ID_TEAM = {
    None: None, # Error
    0: "_FA_", #"Free agent",
    1: "CaDr", #"California Drummers",
    2: "PS", #"Pecs Szentek",
    3: "REc", #"Recanto Dreamers",
    5: "TM5", #"Team 5",
    6: "**BVI**", #"Bela Vista Idles",
    7: "~~SJP~~", #"Saint Joseph Puppers",
}

MAP_STATUS_ABBR = {
    None: None, # Error
    "ACTIVE": "A",
    "QUESTIONABLE": "_Q_",
    "DOUBTFUL": "**D**", # TODO: confirm
    "OUT": "**O**", # TODO: confirm
    "INJURY_RESERVE": "**IR**",
    "DAY_TO_DAY": "DTD"
}

# Base configuration dictionary
class Parameters(TypedDict):
    season: str
    league_id: str
    limit: dict[str, int]
    offset: int
    sort_draft_ranks_type: str
    filter_ranks_slot_ids: list[int]

PARAMETERS = Parameters(
    season = "2026",
    league_id = "503010231",
    limit = {"QB": 20, "RB": 30, "WR": 30, "TE": 20, "D/ST": 15, "K": 15, "HC": 10},
    offset = 0,
    sort_draft_ranks_type = "STANDARD",
    filter_ranks_slot_ids = [0, 2, 4, 6, 17, 16, 8, 9, 10, 12, 13, 24, 11, 14, 15],
)

## Analyzing the API

In [3]:
def find_dict_diff(d1, d2):
    diffs = {}
    for k in set(d1.keys()).union(set(d2.keys())):
        if k not in d1:
            diffs[k] = f"Added in d2: {d2[k]}"
        elif k not in d2:
            diffs[k] = f"Removed in d2: {d1[k]}"
        elif isinstance(d1[k], dict) and isinstance(d2[k], dict):
            diffs.update(find_dict_diff(d1[k], d2[k]))
        elif d1[k] != d2[k]:
            diffs[k] = {"old": d1[k], "new": d2[k]}
    return diffs

In [4]:
queries = {
    "default": '{"players":{"filterStatus":{"value":["FREEAGENT","WAIVERS"]},"filterSlotIds":{"value":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,23,24]},"filterRanksForScoringPeriodIds":{"value":[3]},"sortPercOwned":{"sortPriority":2,"sortAsc":false},"sortDraftRanks":{"sortPriority":100,"sortAsc":true,"value":"STANDARD"},"limit":50,"filterRanksForSlotIds":{"value":[0,2,4,6,17,16,8,9,10,12,13,24,11,14,15]},"filterStatsForTopScoringPeriodIds":{"value":2,"additionalValue":["002026","102026","002025","1120263","022026"]}}}',
    "+QB": '{"players":{"filterStatus":{"value":["FREEAGENT","WAIVERS"]},"filterSlotIds":{"value":[0]},"filterRanksForScoringPeriodIds":{"value":[3]},"limit":50,"offset":0,"sortPercOwned":{"sortAsc":false,"sortPriority":1},"sortDraftRanks":{"sortPriority":100,"sortAsc":true,"value":"STANDARD"},"filterRanksForRankTypes":{"value":["PPR"]},"filterRanksForSlotIds":{"value":[0,2,4,6,17,16,8,9,10,12,13,24,11,14,15]},"filterStatsForTopScoringPeriodIds":{"value":2,"additionalValue":["002026","102026","002025","1120263","022026"]}}}',
    "+all": '{"players":{"filterSlotIds":{"value":[0]},"filterRanksForScoringPeriodIds":{"value":[3]},"limit":50,"offset":0,"sortPercOwned":{"sortAsc":false,"sortPriority":1},"sortDraftRanks":{"sortPriority":100,"sortAsc":true,"value":"STANDARD"},"filterRanksForRankTypes":{"value":["PPR"]},"filterRanksForSlotIds":{"value":[0,2,4,6,17,16,8,9,10,12,13,24,11,14,15]},"filterStatsForTopScoringPeriodIds":{"value":2,"additionalValue":["002026","102026","002025","1120263","022026"]}}}',
    "+sort": '{"players":{"filterSlotIds":{"value":[0]},"filterRanksForScoringPeriodIds":{"value":[3]},"limit":50,"offset":0,"sortAppliedStatTotal":{"sortAsc":false,"sortPriority":1,"value":"1120263"},"sortDraftRanks":{"sortPriority":100,"sortAsc":true,"value":"STANDARD"},"filterRanksForRankTypes":{"value":["PPR"]},"filterRanksForSlotIds":{"value":[0,2,4,6,17,16,8,9,10,12,13,24,11,14,15]},"filterStatsForTopScoringPeriodIds":{"value":2,"additionalValue":["002026","102026","002025","1120263","022026"]}}}',
    "-QB+RB": '{"players":{"filterStatus":{"value":["FREEAGENT","WAIVERS"]},"filterSlotIds":{"value":[2]},"filterRanksForScoringPeriodIds":{"value":[3]},"limit":50,"offset":0,"sortPercOwned":{"sortAsc":false,"sortPriority":1},"sortDraftRanks":{"sortPriority":100,"sortAsc":true,"value":"STANDARD"},"filterRanksForRankTypes":{"value":["PPR"]},"filterRanksForSlotIds":{"value":[0,2,4,6,17,16,8,9,10,12,13,24,11,14,15]},"filterStatsForTopScoringPeriodIds":{"value":2,"additionalValue":["002026","102026","002025","1120263","022026"]}}}',
}

queries_dicts = {k: json.loads(v) for k, v in queries.items()}

In [5]:
queries_dicts["default"]

{'players': {'filterStatus': {'value': ['FREEAGENT', 'WAIVERS']},
  'filterSlotIds': {'value': [0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    23,
    24]},
  'filterRanksForScoringPeriodIds': {'value': [3]},
  'sortPercOwned': {'sortPriority': 2, 'sortAsc': False},
  'sortDraftRanks': {'sortPriority': 100,
   'sortAsc': True,
   'value': 'STANDARD'},
  'limit': 50,
  'filterRanksForSlotIds': {'value': [0,
    2,
    4,
    6,
    17,
    16,
    8,
    9,
    10,
    12,
    13,
    24,
    11,
    14,
    15]},
  'filterStatsForTopScoringPeriodIds': {'value': 2,
   'additionalValue': ['002026', '102026', '002025', '1120263', '022026']}}}

In [6]:
find_dict_diff(queries_dicts["default"], queries_dicts["+QB"])

{'value': {'old': [0,
   1,
   2,
   3,
   4,
   5,
   6,
   7,
   8,
   9,
   10,
   11,
   12,
   13,
   14,
   15,
   16,
   17,
   18,
   19,
   23,
   24],
  'new': [0]},
 'sortPriority': {'old': 2, 'new': 1},
 'filterRanksForRankTypes': "Added in d2: {'value': ['PPR']}",
 'offset': 'Added in d2: 0'}

In [7]:
find_dict_diff(queries_dicts["+QB"], queries_dicts["+all"])

{'filterStatus': "Removed in d2: {'value': ['FREEAGENT', 'WAIVERS']}"}

In [8]:
find_dict_diff(queries_dicts["+all"], queries_dicts["+sort"])

{'sortAppliedStatTotal': "Added in d2: {'sortAsc': False, 'sortPriority': 1, 'value': '1120263'}",
 'sortPercOwned': "Removed in d2: {'sortAsc': False, 'sortPriority': 1}"}

In [9]:
find_dict_diff(queries_dicts["+QB"], queries_dicts["-QB+RB"])

{'value': {'old': [0], 'new': [2]}}

## Extracting the data

In [15]:
def get_players_request_url(
    period: int, position: str, sort_id: int = 0
) -> str:
    stat_types = [
        f"11{PARAMETERS['season']}{period!s}", # Projection for selected week
        f"00{PARAMETERS['season']}", # Realized points until current week
        f"10{PARAMETERS['season']}", # Projection for the whole season
        #f"11{PARAMETERS['season']}{period - 1!s}", # Projection for past (oftencurrent) week
    ]

    x_fantasy_filter = {
        "players": {
            "filterSlotIds": {"value": MAP_POSITION_SLOT[position]},
            "filterRanksForScoringPeriodIds": {"value": [period]},
            "limit": PARAMETERS["limit"][position],
            "offset": PARAMETERS["offset"],
            "sortAppliedStatTotal": {
                "sortAsc": False,
                "sortPriority": 1,
                "value": stat_types[sort_id],
            },
            "sortDraftRanks": {
                "sortPriority": 100,
                "sortAsc": True,
                "value": PARAMETERS["sort_draft_ranks_type"],
            },
            "filterRanksForRankTypes": {"value": ["PPR"]},
            "filterRanksForSlotIds": {"value": PARAMETERS["filter_ranks_slot_ids"]},
            "filterStatsForTopScoringPeriodIds": {
                "value": 2,
                "additionalValue": stat_types,
            },
        }
    }

    s.headers["x-fantasy-filter"] = json.dumps(x_fantasy_filter)
    url = (
        "https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/"
        f"{PARAMETERS['season']}/segments/0/leagues/{PARAMETERS['league_id']}"
        f"?scoringPeriodId={period!s}&view=kona_player_info"
    )
    return url

In [20]:
def get_players_data(
    period: int, position: str, s: requests.Session, sort_id: int = 0
) -> pl.DataFrame:
    stat_ids = [
        f"11{PARAMETERS['season']}{period!s}",
        f"00{PARAMETERS['season']}0",
        f"10{PARAMETERS['season']}0",
        #f"11{PARAMETERS['season']}{period - 1!s}",
    ]

    url = get_players_request_url(
        period = period, position = position, sort_id = sort_id
    )
    response = s.get(url)
    response.raise_for_status()
    data = response.json()

    extracted_records = []
    if "players" not in data:
        return pl.DataFrame(None)

    for entry in data.get("players", []):
        player = entry.get("player", {})
        own = player.get("ownership", {})

        stats = {
            f"{x["statSourceId"]}{x["statSplitTypeId"]}{x["seasonId"]}{x["scoringPeriodId"]}": x
            for x in player.get("stats", [])
        }
        #if not all (stat in stats for stat in stat_ids):
        #    print(f"Missing stats for player {player.get('fullName')}, skipping...")
        #    continue

        extracted_records.append(
            {
                "Name": player.get("fullName"),
                "Team": MAP_ID_TEAM[entry.get("onTeamId")],
                #"Past": stats[stat_ids[3]].get("appliedTotal"),
                "Week": stats[stat_ids[0]].get("appliedTotal"),
                "Rest": (
                    stats[stat_ids[2]].get("appliedTotal", nan) -
                    stats[stat_ids[1]].get("appliedTotal", nan)
                ),
                "Status": MAP_STATUS_ABBR[player.get("injuryStatus")],
                "Start%": f"{own.get("percentStarted", nan):.1f} {own.get("percentChange", nan):+.1f}"
            }
        )
        # CHECK: add if it is a starter in its team, percentOwned

    df = (
        pl.DataFrame(extracted_records)
        .filter(pl.col("Week") > 0)
        .with_columns(pl.col("Week", "Rest").round(1))
    )

    return df

In [12]:
s = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Referer": "https://fantasy.espn.com/",
    "Origin": "https://fantasy.espn.com",
    "x-fantasy-platform": "espn-fantasy-web",
    "x-fantasy-source": "kona",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
}

s = requests.Session()
s.headers.update(headers)

In [16]:
per = 0 # 0 meaning projection for the rest of season
os.makedirs(f"data/week{per}", exist_ok = True)
with open(f"data/week{per}/all.md", "w") as f:
    print("# Strategy\n\n...\n\n", file = f)
    print("# Players", file = f)

In [21]:
for pos in ["QB", "RB", "WR", "TE", "D/ST", "K", "HC"]:
    print(f"Scraping {pos} data...")

    sleep(random() * 10 + 5)
    df = get_players_data(per, pos, s, 2 if per == 0 else 0)

    pos_label = pos if pos != "D/ST" else "DST"
    df.write_csv(f"data/week{per}/{pos_label}.csv")

    df = (
        df
        .join(
            pl.read_csv(f"data/week{per - 1}/{pos_label}.csv")
            .select("Name", Past = "Week"),
            on = "Name", how = "left"
        )
        .select("Name", "Team", "Past", "Week", "Rest", "Status", "Start%")
    )
    if per == 0:
        df = df.sort("Rest", descending = True)

    with (
        pl.Config(
            set_tbl_rows = 999, set_tbl_hide_dataframe_shape = True,
            set_tbl_hide_column_data_types = True, set_tbl_formatting = "MARKDOWN"
        ),
        open(f"data/week{per}/all.md", "a", encoding = "utf-8") as f
    ):
            print(f"\n\n## {pos}\n", file = f)
            print(df, file = f)

Scraping QB data...


KeyError: '1120260'

TODO: run code with sort of projected, then we see if we can get the best players for the rest of the season, if we have some wiggle room in the bench.

Strategy:

1. Optimize starters for current week, adding FAs and oposing benched players better than our best for each position. Avoid dropping players that have good projections for the rest of the season, for small current week gains.
2. Optimize next week. At the beginning of the week, by adding FAs that have good projections for the next week and whole season. Then, closer to the end of week, by adding FAs or opposing benched players that have good projections for the next week.

## Sketches

In [68]:
url = get_players_request_url(period = per, position = "QB")
response = s.get(url)
data = response.json()
data["players"][0]["player"]["stats"]

[{'appliedTotal': 17.46,
  'externalId': '401872953',
  'id': '01401872953',
  'proTeamId': 2,
  'scoringPeriodId': 3,
  'seasonId': 2026,
  'statSourceId': 0,
  'statSplitTypeId': 1,
  'stats': {'0': 26.0,
   '1': 16.0,
   '2': 10.0,
   '3': 204.0,
   '5': 40.0,
   '6': 20.0,
   '7': 10.0,
   '8': 8.0,
   '9': 4.0,
   '10': 2.0,
   '11': 3.0,
   '12': 1.0,
   '13': 2.0,
   '14': 1.0,
   '20': 2.0,
   '21': 61.54,
   '22': 204.0,
   '23': 8.0,
   '24': 22.0,
   '25': 2.0,
   '27': 4.0,
   '28': 2.0,
   '29': 1.0,
   '33': 1.0,
   '39': 2.75,
   '40': 22.0,
   '41': 1.0,
   '42': 1.0,
   '53': 1.0,
   '58': 1.0,
   '59': 9.0,
   '60': 1.0,
   '61': 1.0,
   '64': 4.0,
   '68': 1.0,
   '72': 1.0,
   '73': 3.0,
   '155': 1.0,
   '158': 12.0,
   '179': 2.0,
   '210': 1.0,
   '211': 10.0,
   '212': 3.0}},
 {'appliedTotal': 40.82,
  'externalId': '401872932',
  'id': '01401872932',
  'proTeamId': 2,
  'scoringPeriodId': 2,
  'seasonId': 2026,
  'statSourceId': 0,
  'statSplitTypeId': 1,
  'st

In [69]:
stats = data["players"][0]["player"]["stats"]

stat_ids = [
    f"11{PARAMETERS['season']}{per!s}",
    f"00{PARAMETERS['season']}0",
    f"10{PARAMETERS['season']}0",
    f"11{PARAMETERS['season']}{per - 1!s}",
]
print(stat_ids)

{
    f"{x["statSourceId"]}{x["statSplitTypeId"]}{x["seasonId"]}{x["scoringPeriodId"]}": x
    for x in stats
}.keys()

['1120264', '0020260', '1020260', '1120263']


dict_keys(['0120263', '0120262', '1120264', '0020260', '1020260'])

In [72]:
get_players_data(per, "QB", s)

Name,Team,Week,Rest,Status,Start%
str,str,f64,f64,str,str
"""Josh Allen""","""TM5""",22.3,222.1,"""_Q_""","""98.1 +0.1"""
"""Lamar Jackson""","""REc""",21.1,223.1,"""A""","""94.0 +0.1"""
"""Bryce Young""","""**BVI**""",20.1,167.6,"""A""","""52.7 +1.8"""
"""Kyler Murray""","""TM5""",19.3,237.7,"""A""","""22.5 +0.1"""
"""Tyler Shough""","""**BVI**""",18.9,190.4,"""A""","""52.1 +2.4"""
…,…,…,…,…,…
"""Matthew Stafford""","""PS""",16.4,193.1,"""A""","""40.6 +0.1"""
"""Sam Darnold""","""_FA_""",16.1,186.4,"""A""","""16.1 +10.5"""
"""Jacoby Brissett""","""_FA_""",15.9,166.3,"""A""","""1.6 +1.8"""
